In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/common.ipynb').is_file())
get_ipython().run_line_magic('run', '-i "' + (ROOT / 'src/common.ipynb').as_posix() + '"')
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
from scipy.stats import spearmanr
from pyproj import Transformer
selected_parameters = select_xgb_parameters()


In [ ]:
data = load_data('ALL')
transformer = Transformer.from_crs('EPSG:4326', 'EPSG:2163', always_xy=True)
xy = np.array(transformer.transform(data.LONGITUDE.to_numpy(), data.LATITUDE.to_numpy())).T / 1000
assert xy.shape == (len(data), 2) and np.isfinite(xy).all()
clustering = KMeans(n_clusters=5, random_state=SEED, n_init=50).fit(xy)
order = sorted(range(5), key=lambda k: tuple(clustering.cluster_centers_[k]))
group_ids = {old: new + 1 for new, old in enumerate(order)}
groups = data[['GEOID', 'state', 'LONGITUDE', 'LATITUDE']].copy()
groups['x_km'], groups['y_km'] = xy[:, 0], xy[:, 1]
groups['spatial_group'] = [group_ids[k] for k in clustering.labels_]
assert groups.GEOID.is_unique and set(groups.spatial_group) == {1, 2, 3, 4, 5}


In [ ]:
all_predictions, all_shap = {}, {}
for code in MODELS:
    data = load_data(code)
    group = groups.set_index('GEOID').loc[data.GEOID, 'spatial_group'].to_numpy()
    predicted = np.full(len(data), np.nan)
    contributions = np.full((len(data), len(FEATURES) + 1), np.nan)
    for fold in range(1, 6):
        test, train = group == fold, group != fold
        assert not set(data.loc[train, 'GEOID']) & set(data.loc[test, 'GEOID'])
        model = xgb.XGBRegressor(**selected_parameters)
        model.fit(data.loc[train, FEATURES], data.loc[train, 'EB_pct'])
        predicted[test] = model.predict(data.loc[test, FEATURES])
        contributions[test] = model.get_booster().predict(xgb.DMatrix(data.loc[test, FEATURES]), pred_contribs=True)
        print(f"{code} fold {fold}/5 completed", flush=True)
    assert np.isfinite(predicted).all() and np.isfinite(contributions).all()
    assert np.max(np.abs(contributions.sum(axis=1) - predicted)) < 5e-5
    predictions = data[['GEOID', 'state', 'LONGITUDE', 'LATITUDE', 'EB_pct']].rename(columns={'EB_pct': 'observed'}).copy()
    predictions['group'], predictions['predicted'] = group, predicted
    shap_values = pd.DataFrame(contributions[:, :-1], columns=FEATURES)
    shap_values.insert(0, 'GEOID', data.GEOID.to_numpy())
    shap_values['bias'], shap_values['group'] = contributions[:, -1], group
    all_predictions[code], all_shap[code] = predictions, shap_values
    print(code, 'completed')


In [ ]:
rows = []
for code in MODELS:
    predictions, shap_values = all_predictions[code], all_shap[code]
    assert predictions.GEOID.is_unique and predictions.GEOID.equals(shap_values.GEOID)
    assert set(predictions.GEOID) == set(groups.GEOID)
    assert np.array_equal(predictions.group, shap_values.group)
    assert np.isfinite(predictions[['observed', 'predicted']]).all().all()
    importance = shap_values[FEATURES].abs().mean()
    fitted = fit_global(code)
    background = fitted['data'].iloc[fitted['test_indices']][FEATURES]
    baseline_values = fitted['model'].get_booster().predict(xgb.DMatrix(background), pred_contribs=True)[:, :-1].astype(np.float64)
    original = pd.Series(np.abs(baseline_values).mean(axis=0), index=FEATURES)
    assert original.notna().all() and importance.notna().all()
    overlap = len(set(original.nlargest(5).index) & set(importance.nlargest(5).index))
    rows.append({'Model': SEASON_NAMES[code],
        'R2': r2_score(predictions.observed, predictions.predicted),
        'RMSE (pp)': np.sqrt(mean_squared_error(predictions.observed, predictions.predicted)),
        'MAPE (%)': 100 * mean_absolute_percentage_error(predictions.observed, predictions.predicted),
        'SHAP rank correlation': spearmanr(original.rank(), importance.rank()).statistic,
        'Top-five overlap': f'{overlap}/5'})
table_s7 = pd.DataFrame(rows)
export_table('TableS7', table_s7.round(2))
display(table_s7.round(2))
